
# Household Electricity Consumption Analysis and Forecasting

## 1. Project Introduction

### 1.1 Background

Understanding household electricity consumption is essential for effective energy management, demand forecasting, and power system planning. Electricity demand varies over time due to household activities, appliance usage, and daily or seasonal consumption patterns.

Analyzing historical electricity measurements can reveal these patterns and support the development of data-driven forecasting models.

### 1.2 Project Aim

The aim of this project is to analyze historical household electricity consumption data, identify temporal consumption patterns, and develop machine learning models for short-term electricity demand forecasting.

### 1.3 Research Objectives

The specific objectives are to:

1. Examine the structure, quality, and characteristics of household electricity consumption data.
2. Identify and address missing values, inconsistencies, and other data quality issues.
3. Investigate hourly, daily, weekly, and seasonal electricity consumption patterns.
4. Explore relationships between electrical measurements and household energy consumption.
5. Develop time-based, lagged, and rolling statistical features for forecasting.
6. Build and compare baseline and machine learning forecasting models.
7. Evaluate forecasting performance using appropriate error metrics.
8. Interpret the findings and discuss their implications for household energy management.

### 1.4 Dataset Overview

This project uses the **Individual Household Electric Power Consumption** dataset from the UCI Machine Learning Repository.

The dataset contains approximately 2.08 million measurements recorded at one-minute intervals between December 2006 and November 2010.

The measurements include active power, reactive power, voltage, current intensity, and three household energy sub-metering variables.

**Dataset source:** https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

### 1.5 Project Methodology

The analysis follows eight major phases:

1. Project planning and dataset selection.
2. Project setup and version control.
3. Data loading and quality assessment.
4. Data cleaning and exploratory data analysis.
5. Time-series analysis and feature engineering.
6. Electricity demand forecasting.
7. Model evaluation and interpretation.
8. Final documentation and reproducibility review.

To avoid data leakage, forecasting models will be evaluated using chronological training and testing periods rather than random splitting.



### 1.6 Project Workflow Animation

The following animation illustrates the five main stages of the household electricity consumption analysis and forecasting workflow.

![Household Electricity Consumption and Forecasting Workflow](../figure/household_electricity_project_workflow.gif)

**Note:** The animation is a conceptual illustration of the project workflow. The electricity consumption patterns shown are illustrative rather than actual dataset results.



## 2. Python Environment and Library Imports

### 2.1 Import Required Libraries

This section imports the Python libraries required for data manipulation, numerical analysis, visualization, and file management.

Additional libraries for machine learning and forecasting will be introduced in later sections when needed.


In [1]:

# Import libraries for data analysis and visualization

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

# Configure default visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("All required libraries imported successfully!")


All required libraries imported successfully!



### 2.2 Define Project Directories

To ensure reproducibility and maintain a well-organized project, the dataset and figure directories are defined using Python's `pathlib` library.

The notebook is stored in the `notebook/` directory, while the raw dataset is stored in `data/`. Generated visualizations will be saved in `figure/`.

Relative paths are used to avoid hardcoding computer-specific directory locations.


In [2]:

# Define the main project directory
PROJECT_DIR = Path.cwd().parent

# Define dataset and figure directories
DATA_DIR = PROJECT_DIR / "data"
FIGURE_DIR = PROJECT_DIR / "figure"

# Define the dataset file path
DATA_FILE = DATA_DIR / "household_power_consumption.txt"

# Check whether the dataset exists
print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Figure directory:", FIGURE_DIR)
print("Dataset exists:", DATA_FILE.is_file())


Project directory: C:\Users\USER\Desktop\ECE M2\M2 Sem 1\BIG DATA & ENERGY DATA ANALYSIS\Personal Projects\Household Electricity Consumption Analysis and Forecasting
Data directory: C:\Users\USER\Desktop\ECE M2\M2 Sem 1\BIG DATA & ENERGY DATA ANALYSIS\Personal Projects\Household Electricity Consumption Analysis and Forecasting\data
Figure directory: C:\Users\USER\Desktop\ECE M2\M2 Sem 1\BIG DATA & ENERGY DATA ANALYSIS\Personal Projects\Household Electricity Consumption Analysis and Forecasting\figure
Dataset exists: True



## 3. Data Loading and Initial Inspection

### 3.1 Load the Household Electricity Consumption Dataset

The dataset is loaded using Pandas. The original file uses semicolons (`;`) to separate variables and question marks (`?`) to represent missing measurements.

Missing-value indicators are converted into `NaN` values during import to facilitate subsequent data quality assessment and preprocessing.

The original date and time columns are preserved at this stage. Timestamp conversion and validation will be performed separately to allow systematic examination of the raw data.


In [3]:

# Load the household electricity consumption dataset

df = pd.read_csv(
    DATA_FILE,
    sep=";",
    na_values=["?"],
    low_memory=False
)

# Confirm successful loading
print("Dataset loaded successfully!")

# Display dataset dimensions
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

# Display the first five observations
display(df.head())


Dataset loaded successfully!
Number of rows: 2,075,259
Number of columns: 9


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0



### 3.2 Inspect Dataset Structure and Data Types

This section examines the structure of the loaded dataset, including its column names, data types, non-null observations, and memory usage.

The inspection helps identify variables that may require conversion or preprocessing before conducting time-series analysis.

In particular, the `Date` and `Time` variables must eventually be combined into a datetime representation suitable for chronological analysis.


In [4]:

# Inspect dataset structure and data types

print("Dataset Information:")
df.info()

print("\nColumn Data Types:")
print(df.dtypes)

print("\nDataset Memory Usage (MB):")
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"{memory_mb:.2f} MB")


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), object(2)
memory usage: 142.5+ MB

Column Data Types:
Date                      object
Time                      object
Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object

Dataset Memory Usage (MB):
338.33 MB



### 3.3 Missing Value Assessment

Missing measurements can affect statistical analysis, time-series continuity, and forecasting accuracy.

This section quantifies missing values in each variable and calculates their percentages relative to the total number of observations.

The assessment is performed before data cleaning to preserve an accurate record of the original dataset quality.

No missing values will be removed or imputed at this stage.


In [5]:

# Calculate missing values for each column
missing_count = df.isna().sum()

# Calculate missing percentages
missing_percentage = (missing_count / len(df)) * 100

# Create a summary table
missing_summary = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage (%)": missing_percentage
})

# Sort columns by number of missing values
missing_summary = missing_summary.sort_values(
    by="Missing Values",
    ascending=False
)

# Display the summary
print("Missing Value Assessment:")
display(missing_summary.round(2))

# Count observations containing at least one missing value
rows_with_missing = df.isna().any(axis=1).sum()

print(f"\nRows containing missing values: {rows_with_missing:,}")
print(f"Percentage of affected rows: {rows_with_missing / len(df) * 100:.2f}%")


Missing Value Assessment:


,Missing Values,Missing Percentage (%)
Global_active_power,25979,1.25
Global_reactive_power,25979,1.25
Voltage,25979,1.25
Global_intensity,25979,1.25
Sub_metering_1,25979,1.25
Sub_metering_2,25979,1.25
Sub_metering_3,25979,1.25
Date,0,0.00
Time,0,0.00



Rows containing missing values: 25,979
Percentage of affected rows: 1.25%



### 3.4 Missing-Value Patterns and Continuity

Although missing values represent approximately 1.25% of the dataset, their distribution over time must be examined before selecting a cleaning strategy.

This section investigates whether missing electrical measurements occur simultaneously across variables and whether they form consecutive sequences.

Understanding these patterns will help determine appropriate missing-data treatment methods during preprocessing.


In [6]:

# Identify the seven electrical measurement columns
measurement_cols = df.columns.drop(["Date", "Time"])

# Identify rows where all electrical measurements are missing
all_missing = df[measurement_cols].isna().all(axis=1)

# Identify rows where at least one electrical measurement is missing
any_missing = df[measurement_cols].isna().any(axis=1)

print("Missing-Value Pattern Assessment")
print("-" * 40)

print(f"Rows with all measurements missing: {all_missing.sum():,}")
print(f"Rows with any measurement missing: {any_missing.sum():,}")

# Identify consecutive groups of missing observations
missing_groups = all_missing.ne(all_missing.shift()).cumsum()

# Calculate lengths of consecutive missing sequences
missing_run_lengths = (
    all_missing.groupby(missing_groups)
    .sum()
)

# Keep only sequences containing missing observations
missing_run_lengths = missing_run_lengths[missing_run_lengths > 0]

print(f"\nNumber of missing sequences: {len(missing_run_lengths):,}")

if not missing_run_lengths.empty:
    print(f"Longest missing sequence: {missing_run_lengths.max():,} rows")
    print(f"Median missing sequence: {missing_run_lengths.median():.1f} rows")
else:
    print("No missing sequences detected.")


Missing-Value Pattern Assessment
----------------------------------------
Rows with all measurements missing: 25,979
Rows with any measurement missing: 25,979

Number of missing sequences: 71
Longest missing sequence: 7,226 rows
Median missing sequence: 1.0 rows



### 3.5 Date and Time Validation

Accurate timestamps are essential for identifying electricity consumption patterns and developing reliable forecasting models.

This section combines the original `Date` and `Time` columns into a datetime representation and examines the resulting timestamps.

The dataset uses the day/month/year date format. Invalid timestamps will be identified without removing or modifying the original observations.

Timestamp continuity and duplicate timestamps will be assessed separately.


In [7]:

# Combine the Date and Time columns
datetime_text = df["Date"] + " " + df["Time"]

# Convert the combined text into datetime values
df["Datetime"] = pd.to_datetime(
    datetime_text,
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

# Assess datetime conversion
invalid_timestamps = df["Datetime"].isna().sum()

print("Date and Time Validation")
print("-" * 40)

print(f"Invalid timestamps: {invalid_timestamps:,}")
print(f"Earliest timestamp: {df['Datetime'].min()}")
print(f"Latest timestamp: {df['Datetime'].max()}")

print("\nDatetime column type:")
print(df["Datetime"].dtype)

# Display the first five converted timestamps
display(df[["Date", "Time", "Datetime"]].head())


Date and Time Validation
----------------------------------------
Invalid timestamps: 0
Earliest timestamp: 2006-12-16 17:24:00
Latest timestamp: 2010-11-26 21:02:00

Datetime column type:
datetime64[ns]


,Date,Time,Datetime
0,16/12/2006,17:24:00,2006-12-16 17:24:00
1,16/12/2006,17:25:00,2006-12-16 17:25:00
2,16/12/2006,17:26:00,2006-12-16 17:26:00
3,16/12/2006,17:27:00,2006-12-16 17:27:00
4,16/12/2006,17:28:00,2006-12-16 17:28:00



### 3.6 Timestamp Continuity and Duplicate Assessment

This section evaluates the chronological integrity of the electricity consumption dataset.

The assessment checks for duplicate timestamps, chronological ordering, and deviations from the expected one-minute measurement frequency.

Identifying timestamp inconsistencies is essential for reliable time-series analysis, energy aggregation, and forecasting.

No observations will be removed or modified during this assessment.


In [8]:

# Check for duplicate timestamps
duplicate_timestamps = df["Datetime"].duplicated().sum()

# Check whether timestamps are chronologically ordered
is_chronological = df["Datetime"].is_monotonic_increasing

# Calculate differences between consecutive timestamps
time_differences = df["Datetime"].diff()

# Define the expected measurement interval
expected_interval = pd.Timedelta(minutes=1)

# Count intervals that differ from one minute
irregular_intervals = (
    time_differences.iloc[1:] != expected_interval
).sum()

# Calculate the expected number of one-minute timestamps
expected_rows = int(
    (df["Datetime"].max() - df["Datetime"].min())
    / expected_interval
) + 1

# Display results
print("Timestamp Continuity Assessment")
print("-" * 40)

print(f"Duplicate timestamps: {duplicate_timestamps:,}")
print(f"Chronologically ordered: {is_chronological}")
print(f"Irregular time intervals: {irregular_intervals:,}")
print(f"Expected one-minute timestamps: {expected_rows:,}")
print(f"Actual observations: {len(df):,}")
print(f"Difference: {expected_rows - len(df):,}")


Timestamp Continuity Assessment
----------------------------------------
Duplicate timestamps: 0
Chronologically ordered: True
Irregular time intervals: 0
Expected one-minute timestamps: 2,075,259
Actual observations: 2,075,259
Difference: 0



## 4. Data Cleaning and Preprocessing

This section prepares the household electricity consumption dataset for exploratory data analysis and forecasting.

The initial data quality assessment identified 25,979 observations (1.25%) with missing electrical measurements, distributed across 71 consecutive sequences. The longest sequence contains 7,226 observations.

Although the dataset contains a complete sequence of one-minute timestamps, missing electrical measurements require careful treatment.

The preprocessing strategy will prioritize:

1. Preserving the original dataset for reproducibility.
2. Avoiding artificial electricity consumption patterns.
3. Distinguishing short measurement gaps from extended missing periods.
4. Maintaining chronological integrity.
5. Preventing data leakage during forecasting.

Missing-value treatment decisions will be evaluated before applying them to the working dataset.



### 4.1 Create a Working Copy of the Dataset

To preserve the original dataset, a separate working DataFrame is created for data cleaning and preprocessing.

This approach allows preprocessing operations to be performed without modifying the original loaded data. It also supports reproducibility and enables comparisons between the raw and processed datasets.

The working copy retains all observations, electrical measurements, and timestamps at this stage.


In [9]:

# Create an independent working copy of the dataset
df_clean = df.copy()

# Compare the original and working datasets
print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

# Verify that the working copy is a separate DataFrame
print("Independent working copy:", df_clean is not df)

# Check missing values before preprocessing
print(f"Original missing rows: {df.isna().any(axis=1).sum():,}")
print(f"Working copy missing rows: {df_clean.isna().any(axis=1).sum():,}")

# Preview the working dataset
display(df_clean.head())


Original dataset shape: (2075259, 10)
Working dataset shape: (2075259, 10)
Independent working copy: True
Original missing rows: 25,979
Working copy missing rows: 25,979


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00



### 4.2 Missing-Value Treatment Strategy

The dataset contains 25,979 observations (1.25%) with missing electrical measurements, distributed across 71 consecutive sequences.

Because the timestamp assessment confirmed uninterrupted one-minute intervals, the lengths of consecutive missing sequences can be interpreted as durations in minutes.

A differentiated treatment strategy is considered:

1. **Short gaps (1–5 minutes):** Potentially suitable for interpolation during exploratory analysis, subject to validation.
2. **Medium gaps (6–60 minutes):** Require additional investigation before imputation.
3. **Long gaps (more than 60 minutes):** Avoid automatic interpolation because extended gaps may introduce artificial consumption patterns.

For forecasting, missing-data treatment must be performed without using future observations that would be unavailable at prediction time.

The original missing values will be preserved until the treatment strategy has been evaluated.


In [10]:

# Classify missing sequences according to their lengths

short_gaps = (missing_run_lengths <= 5).sum()

medium_gaps = (
    (missing_run_lengths > 5) &
    (missing_run_lengths <= 60)
).sum()

long_gaps = (missing_run_lengths > 60).sum()

# Calculate missing observations in each category
short_missing_rows = missing_run_lengths[
    missing_run_lengths <= 5
].sum()

medium_missing_rows = missing_run_lengths[
    (missing_run_lengths > 5) &
    (missing_run_lengths <= 60)
].sum()

long_missing_rows = missing_run_lengths[
    missing_run_lengths > 60
].sum()

# Create a summary table
gap_summary = pd.DataFrame({
    "Gap Category": [
        "Short (1–5 minutes)",
        "Medium (6–60 minutes)",
        "Long (>60 minutes)"
    ],
    "Number of Gaps": [
        short_gaps,
        medium_gaps,
        long_gaps
    ],
    "Missing Observations": [
        short_missing_rows,
        medium_missing_rows,
        long_missing_rows
    ]
})

print("Missing-Value Gap Classification:")
display(gap_summary)

print(
    f"\nTotal missing observations classified: "
    f"{gap_summary['Missing Observations'].sum():,}"
)


Missing-Value Gap Classification:


,Gap Category,Number of Gaps,Missing Observations
0,Short (1–5 minutes),55,76
1,Medium (6–60 minutes),7,212
2,Long (>60 minutes),9,25691



Total missing observations classified: 25,979



### 4.3 Limited Interpolation of Short Missing-Value Gaps

The missing-value assessment identified 55 short gaps (1–5 minutes), containing 76 missing observations.

For exploratory data analysis, linear interpolation will be applied only to complete missing sequences of five minutes or fewer. Medium and long gaps will remain missing to avoid generating artificial consumption patterns across extended periods.

A separate exploratory DataFrame will be created to preserve the original and working datasets.

This interpolation uses observations on both sides of a gap and is therefore intended only for retrospective exploratory analysis. Forecasting preprocessing will be handled separately to prevent future-information leakage.


In [11]:

# Create a separate dataset for exploratory analysis
df_eda = df_clean.copy()

# Identify electrical measurement columns
measurement_cols = df_clean.columns.drop(
    ["Date", "Time", "Datetime"]
)

# Identify rows belonging to short missing sequences
short_gap_groups = missing_run_lengths[
    missing_run_lengths <= 5
].index

short_gap_mask = (
    all_missing &
    missing_groups.isin(short_gap_groups)
)

# Interpolate using surrounding valid measurements
interpolated_values = df_clean[measurement_cols].interpolate(
    method="linear",
    limit_area="inside"
)

# Replace values only within identified short gaps
df_eda.loc[short_gap_mask, measurement_cols] = (
    interpolated_values.loc[short_gap_mask, measurement_cols]
)

# Compare missing observations before and after interpolation
missing_before = df_clean[measurement_cols].isna().any(axis=1).sum()
missing_after = df_eda[measurement_cols].isna().any(axis=1).sum()

print("Limited Interpolation Assessment")
print("-" * 40)
print(f"Missing rows before: {missing_before:,}")
print(f"Missing rows after: {missing_after:,}")
print(f"Rows successfully filled: {missing_before - missing_after:,}")

# Verify original working copy remains unchanged
print(
    "Original working copy missing rows:",
    df_clean[measurement_cols].isna().any(axis=1).sum()
)


Limited Interpolation Assessment
----------------------------------------
Missing rows before: 25,979
Missing rows after: 25,903
Rows successfully filled: 76
Original working copy missing rows: 25979



### 4.4 Validation of the Interpolated Dataset

Following limited interpolation, the processed dataset is validated to ensure that only the intended short missing-value gaps were modified.

The validation checks whether:

1. The dataset dimensions remain unchanged.
2. Previously valid measurements retain their original values.
3. Only the 76 identified short-gap observations were filled.
4. Medium and long missing sequences remain unfilled.
5. No negative electrical measurements were introduced.

These checks help ensure that preprocessing preserves the integrity of the original electricity consumption measurements.


In [12]:

# Validate the exploratory dataset after limited interpolation

# 1. Confirm dataset dimensions are unchanged
shape_unchanged = df_eda.shape == df_clean.shape

# 2. Confirm originally valid measurements are unchanged
originally_valid = df_clean[measurement_cols].notna()

valid_values_unchanged = (
    df_eda[measurement_cols]
    .where(originally_valid)
    .equals(df_clean[measurement_cols].where(originally_valid))
)

# 3. Identify rows where missing values were filled
filled_mask = (
    df_clean[measurement_cols].isna().any(axis=1)
    & df_eda[measurement_cols].notna().all(axis=1)
)

filled_rows = filled_mask.sum()

# 4. Confirm all filled rows belong to short gaps
only_short_gaps_filled = (
    filled_mask & ~short_gap_mask
).sum() == 0

# 5. Check that no new missing values were introduced
new_missing_values = (
    df_eda[measurement_cols].isna()
    & df_clean[measurement_cols].notna()
).sum().sum()

# 6. Check for negative electrical measurements
negative_values = (
    df_eda[measurement_cols] < 0
).sum().sum()

print("Interpolated Dataset Validation")
print("-" * 45)
print(f"Dataset dimensions unchanged: {shape_unchanged}")
print(f"Originally valid values unchanged: {valid_values_unchanged}")
print(f"Rows filled: {filled_rows:,}")
print(f"Only short gaps filled: {only_short_gaps_filled}")
print(f"New missing values introduced: {new_missing_values:,}")
print(f"Negative measurements detected: {negative_values:,}")


Interpolated Dataset Validation
---------------------------------------------
Dataset dimensions unchanged: True
Originally valid values unchanged: True
Rows filled: 76
Only short gaps filled: True
New missing values introduced: 0
Negative measurements detected: 0



### 4.5 Preparing the Time-Indexed Dataset

Electricity consumption measurements are time-dependent observations. A datetime index is therefore required to support time-series operations, including resampling, temporal aggregation, and the analysis of hourly, daily, weekly, and seasonal consumption patterns.

The validated exploratory dataset is copied into a time-indexed DataFrame, preserving the original dataset and the preprocessing results.

The timestamp index is checked for chronological ordering, uniqueness, and missing values before further analysis.


In [13]:

# Create a time-indexed dataset for exploratory analysis
df_time = df_eda.copy()

# Set Datetime as the index
df_time = df_time.set_index("Datetime")

# Ensure observations are chronologically ordered
df_time = df_time.sort_index()

# Validate the datetime index
print("Time-Indexed Dataset Validation")
print("-" * 45)

print(f"Dataset shape: {df_time.shape}")
print(f"Index type: {type(df_time.index).__name__}")
print(f"Chronologically ordered: {df_time.index.is_monotonic_increasing}")
print(f"Unique timestamps: {df_time.index.is_unique}")
print(f"Missing timestamps: {df_time.index.isna().sum():,}")

print(f"\nFirst timestamp: {df_time.index.min()}")
print(f"Last timestamp: {df_time.index.max()}")

# Display the first five time-indexed observations
display(df_time.head())


Time-Indexed Dataset Validation
---------------------------------------------
Dataset shape: (2075259, 9)
Index type: DatetimeIndex
Chronologically ordered: True
Unique timestamps: True
Missing timestamps: 0

First timestamp: 2006-12-16 17:24:00
Last timestamp: 2010-11-26 21:02:00


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,,,
2006-12-16 17:24:00,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0



### 4.6 Hourly Electricity Consumption Aggregation

The household electricity dataset contains minute-level active power measurements expressed in kilowatts (kW).

To estimate hourly electricity consumption in kilowatt-hours (kWh), minute-level power measurements are summed and divided by 60.

An hourly observation is considered complete only when all 60 minute-level measurements are available.

Incomplete hours are retained in the aggregated dataset but assigned missing hourly energy values. This avoids treating partial measurements as complete hourly electricity consumption.

The resulting hourly dataset will support exploratory analysis of electricity demand and subsequent forecasting.


In [14]:

# Aggregate minute-level active power into hourly energy consumption

# Extract active power measurements (kW)
active_power = df_time["Global_active_power"]

# Count available measurements in each hour
hourly_valid_minutes = active_power.resample("h").count()

# Convert minute-level power readings to hourly energy (kWh)
hourly_energy = active_power.resample("h").sum(min_count=60) / 60

# Create the hourly analysis dataset
df_hourly = pd.DataFrame({
    "Energy_kWh": hourly_energy,
    "Valid_Minutes": hourly_valid_minutes
})

# Identify complete and incomplete hours
df_hourly["Complete_Hour"] = df_hourly["Valid_Minutes"] == 60

# Summarize the aggregation
print("Hourly Electricity Consumption Aggregation")
print("-" * 50)

print(f"Total hourly intervals: {len(df_hourly):,}")
print(f"Complete hours: {df_hourly['Complete_Hour'].sum():,}")
print(f"Incomplete hours: {(~df_hourly['Complete_Hour']).sum():,}")
print(f"Missing hourly energy values: {df_hourly['Energy_kWh'].isna().sum():,}")

print("\nFirst five hourly observations:")
display(df_hourly.head())


Hourly Electricity Consumption Aggregation
--------------------------------------------------
Total hourly intervals: 34,589
Complete hours: 34,139
Incomplete hours: 450
Missing hourly energy values: 450

First five hourly observations:


,Energy_kWh,Valid_Minutes,Complete_Hour
Datetime,,,
2006-12-16 17:00:00,NaN,36,False
2006-12-16 18:00:00,3.632200,60,True
2006-12-16 19:00:00,3.400233,60,True
2006-12-16 20:00:00,3.268567,60,True
2006-12-16 21:00:00,3.056467,60,True



### 4.7 Validation of Hourly Electricity Consumption

The hourly electricity consumption dataset is validated before exploratory analysis.

The validation checks that:

1. The total number of hourly intervals equals the sum of complete and incomplete hours.
2. Every complete hour contains exactly 60 valid minute-level measurements.
3. Incomplete hours do not contain calculated energy values.
4. Complete hours contain valid, non-negative energy values.
5. Hourly energy calculations are consistent with the original minute-level power measurements.

These checks ensure that the aggregated dataset is suitable for investigating electricity consumption patterns and developing forecasting models.


In [15]:

# Validate hourly electricity consumption aggregation

# 1. Check interval classification
intervals_consistent = (
    df_hourly["Complete_Hour"].sum()
    + (~df_hourly["Complete_Hour"]).sum()
    == len(df_hourly)
)

# 2. Check complete-hour measurement counts
complete_hours_valid = (
    df_hourly.loc[
        df_hourly["Complete_Hour"], "Valid_Minutes"
    ] == 60
).all()

# 3. Check incomplete hours have missing energy values
incomplete_hours_missing = (
    df_hourly.loc[
        ~df_hourly["Complete_Hour"], "Energy_kWh"
    ].isna().all()
)

# 4. Check complete hours have valid, non-negative energy
complete_energy = df_hourly.loc[
    df_hourly["Complete_Hour"], "Energy_kWh"
]

valid_complete_energy = (
    complete_energy.notna().all()
    and (complete_energy >= 0).all()
)

# 5. Independently calculate the first complete hour
first_complete_hour = df_hourly.index[
    df_hourly["Complete_Hour"]
][0]

minute_power = df_time.loc[
    first_complete_hour:
    first_complete_hour + pd.Timedelta(minutes=59),
    "Global_active_power"
]

manual_energy = minute_power.sum() / 60

aggregated_energy = df_hourly.loc[
    first_complete_hour, "Energy_kWh"
]

energy_calculation_matches = np.isclose(
    manual_energy, aggregated_energy
)

print("Hourly Energy Validation")
print("-" * 45)
print(f"Hourly intervals consistent: {intervals_consistent}")
print(f"Complete hours contain 60 measurements: {complete_hours_valid}")
print(f"Incomplete hours have missing energy: {incomplete_hours_missing}")
print(f"Complete-hour energy values valid: {valid_complete_energy}")
print(f"Manual energy calculation matches: {energy_calculation_matches}")

print(f"\nFirst complete hour: {first_complete_hour}")
print(f"Manually calculated energy: {manual_energy:.6f} kWh")
print(f"Aggregated energy: {aggregated_energy:.6f} kWh")


Hourly Energy Validation
---------------------------------------------
Hourly intervals consistent: True
Complete hours contain 60 measurements: True
Incomplete hours have missing energy: True
Complete-hour energy values valid: True
Manual energy calculation matches: True

First complete hour: 2006-12-16 18:00:00
Manually calculated energy: 3.632200 kWh
Aggregated energy: 3.632200 kWh
